In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================

from datetime import date, datetime, timezone
from uuid import uuid4

from delta.tables import DeltaTable
from pyspark.sql import functions as F


"""
Build conformed Gold dimensions from validated Silver reference and
operational datasets. Surrogate keys isolate analytical relationships from
source business keys and support historical device classifications.
"""

CATALOG = "semiconplus_portfolio"
GOLD_SCHEMA = "gold"

TABLES = {
    "silver_sites": f"{CATALOG}.silver.sites",
    "silver_product_groups": f"{CATALOG}.silver.product_groups",
    "silver_devices": f"{CATALOG}.silver.devices",
    "silver_equipment": f"{CATALOG}.silver.equipment",
    "silver_tests": f"{CATALOG}.silver.unit_test_results",
    "dim_date": f"{CATALOG}.{GOLD_SCHEMA}.dim_date",
    "dim_site": f"{CATALOG}.{GOLD_SCHEMA}.dim_site",
    "dim_product_group": f"{CATALOG}.{GOLD_SCHEMA}.dim_product_group",
    "dim_equipment": f"{CATALOG}.{GOLD_SCHEMA}.dim_equipment",
    "dim_defect": f"{CATALOG}.{GOLD_SCHEMA}.dim_defect",
    "dim_test_program": f"{CATALOG}.{GOLD_SCHEMA}.dim_test_program",
    "dim_device_scd2": f"{CATALOG}.{GOLD_SCHEMA}.dim_device_scd2",
}

PIPELINE_RUN_ID = str(uuid4())
PIPELINE_START_TIME = datetime.now(timezone.utc)

HISTORY_START_DATE = date(2021, 1, 1)
OPEN_END_DATE = date(9999, 12, 31)
SIMULATED_CHANGE_DATE = date(2026, 1, 1)
SIMULATED_DEVICE_ID = "DV001"

print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Pipeline start UTC: {PIPELINE_START_TIME.isoformat()}")

In [0]:
# ===================================================
# BLOCK 2 — DEPENDENCY CHECKS
# ===================================================


"""
Confirm that all validated Silver dependencies are available before Gold
dimensions are replaced or updated.
"""

required_sources = [
    TABLES["silver_sites"],
    TABLES["silver_product_groups"],
    TABLES["silver_devices"],
    TABLES["silver_equipment"],
    TABLES["silver_tests"],
]

missing_sources = [
    table_name
    for table_name in required_sources
    if not spark.catalog.tableExists(table_name)
]

assert not missing_sources, f"Missing Silver sources: {missing_sources}"

assert spark.table(TABLES["silver_sites"]).count() == 3
assert spark.table(TABLES["silver_product_groups"]).count() == 6
assert spark.table(TABLES["silver_devices"]).count() == 30
assert spark.table(TABLES["silver_equipment"]).count() == 24

print("Gold dimension dependencies passed.")

In [0]:
# ===================================================
# BLOCK 3 — DIMENSION WRITE CONTROL
# ===================================================


"""
Persist a complete dimension snapshot and verify surrogate-key uniqueness.

Snapshot dimensions use deterministic keys so reruns produce stable fact
relationships without duplicate dimension members.
"""

def write_dimension(dimension_df, table_name, surrogate_key):
    duplicate_keys = (
        dimension_df
        .groupBy(surrogate_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_keys == 0, (
        f"Duplicate surrogate keys detected for {table_name}."
    )

    (
        dimension_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    persisted_count = spark.table(table_name).count()
    assert persisted_count == dimension_df.count()

    print(f"Written {table_name}: {persisted_count:,} records")

In [0]:
# ===================================================
# BLOCK 4 — BUILD DATE DIMENSION
# ===================================================

"""
Create one calendar member per day across the complete historical dataset
and the controlled SCD demonstration period.
"""

date_rows_df = spark.sql(
    """
    SELECT EXPLODE(
        SEQUENCE(
            TO_DATE('2021-01-01'),
            TO_DATE('2026-12-31'),
            INTERVAL 1 DAY
        )
    ) AS calendar_date
    """
)

dim_date_df = date_rows_df.select(
    F.date_format("calendar_date", "yyyyMMdd").cast("int").alias("date_key"),
    "calendar_date",
    F.year("calendar_date").alias("calendar_year"),
    F.quarter("calendar_date").alias("calendar_quarter"),
    F.month("calendar_date").alias("calendar_month"),
    F.date_format("calendar_date", "MMMM").alias("month_name"),
    F.weekofyear("calendar_date").alias("week_of_year"),
    F.dayofmonth("calendar_date").alias("day_of_month"),
    F.dayofweek("calendar_date").alias("day_of_week"),
    F.date_format("calendar_date", "EEEE").alias("day_name"),
    F.when(F.dayofweek("calendar_date").isin(1, 7), True)
    .otherwise(False)
    .alias("is_weekend"),
    F.date_trunc("month", "calendar_date").cast("date").alias("month_start_date"),
    F.last_day("calendar_date").alias("month_end_date"),
    F.lit(PIPELINE_RUN_ID).alias("_gold_pipeline_run_id"),
    F.current_timestamp().alias("_gold_processed_at_utc"),
)

unknown_date_df = spark.createDataFrame(
    [(0, None, 0, 0, 0, "Unknown", 0, 0, 0, "Unknown", False, None, None)],
    """
    date_key INT, calendar_date DATE, calendar_year INT,
    calendar_quarter INT, calendar_month INT, month_name STRING,
    week_of_year INT, day_of_month INT, day_of_week INT,
    day_name STRING, is_weekend BOOLEAN, month_start_date DATE,
    month_end_date DATE
    """,
).withColumn(
    "_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID)
).withColumn(
    "_gold_processed_at_utc", F.current_timestamp()
)

dim_date_df = unknown_date_df.unionByName(dim_date_df)

write_dimension(dim_date_df, TABLES["dim_date"], "date_key")

In [0]:
# ===================================================
# BLOCK 5 — BUILD SITE DIMENSION
# ===================================================

"""
Create the conformed manufacturing-site dimension with stable surrogate
keys and local-timezone attributes.
"""

site_members_df = (
    spark.table(TABLES["silver_sites"])
    .select(
        F.xxhash64("site_id").alias("site_key"),
        "site_id",
        "site_name",
        "country",
        "timezone",
    )
)

unknown_site_df = spark.createDataFrame(
    [(0, "UNKNOWN", "Unknown Site", "Unknown", "UTC")],
    "site_key LONG, site_id STRING, site_name STRING, country STRING, timezone STRING",
)

dim_site_df = (
    unknown_site_df.unionByName(site_members_df)
    .withColumn("_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

write_dimension(dim_site_df, TABLES["dim_site"], "site_key")

In [0]:
# ===================================================
# BLOCK 6 — BUILD PRODUCT-GROUP DIMENSION
# ===================================================

"""
Create the product-group dimension including business-unit and restriction
attributes used by downstream governance views.
"""

product_group_members_df = (
    spark.table(TABLES["silver_product_groups"])
    .select(
        F.xxhash64("product_group_id").alias("product_group_key"),
        "product_group_id",
        "product_group_name",
        "business_unit",
        "restricted",
    )
)

unknown_product_group_df = spark.createDataFrame(
    [(0, "UNKNOWN", "Unknown Product Group", "Unknown", True)],
    """
    product_group_key LONG, product_group_id STRING,
    product_group_name STRING, business_unit STRING, restricted BOOLEAN
    """,
)

dim_product_group_df = (
    unknown_product_group_df.unionByName(product_group_members_df)
    .withColumn("_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

write_dimension(
    dim_product_group_df,
    TABLES["dim_product_group"],
    "product_group_key",
)

In [0]:
# ===================================================
# BLOCK 7 — BUILD EQUIPMENT DIMENSION
# ===================================================

"""
Create the equipment dimension and resolve each equipment member to its
conformed site surrogate key.
"""

site_key_df = dim_site_df.select("site_key", "site_id")

equipment_members_df = (
    spark.table(TABLES["silver_equipment"])
    .join(site_key_df, "site_id", "left")
    .select(
        F.xxhash64("equipment_id").alias("equipment_key"),
        "equipment_id",
        F.coalesce("site_key", F.lit(0)).alias("site_key"),
        "site_id",
        "equipment_model",
        "equipment_type",
        "rated_units_per_hour",
    )
)

unknown_equipment_df = spark.createDataFrame(
    [(0, "UNKNOWN", 0, "UNKNOWN", "Unknown", "UNKNOWN", 0)],
    """
    equipment_key LONG, equipment_id STRING, site_key LONG,
    site_id STRING, equipment_model STRING, equipment_type STRING,
    rated_units_per_hour INT
    """,
)

dim_equipment_df = (
    unknown_equipment_df.unionByName(equipment_members_df)
    .withColumn("_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

write_dimension(
    dim_equipment_df,
    TABLES["dim_equipment"],
    "equipment_key",
)

In [0]:
# ===================================================
# BLOCK 8 — BUILD DEFECT DIMENSION
# ===================================================

"""
Create a conformed defect dimension from validated unit-test outcomes.
PASS is retained as a non-defect classification for complete test-result
relationships.
"""

defect_members_df = (
    spark.table(TABLES["silver_tests"])
    .select("defect_code")
    .filter(F.col("defect_code").isNotNull())
    .distinct()
    .select(
        F.xxhash64("defect_code").alias("defect_key"),
        "defect_code",
        F.when(F.col("defect_code") == "PASS", "PASS")
        .otherwise("FAILURE")
        .alias("defect_category"),
        (F.col("defect_code") != "PASS").alias("is_defect"),
    )
)

unknown_defect_df = spark.createDataFrame(
    [(0, "UNKNOWN", "UNKNOWN", False)],
    """
    defect_key LONG, defect_code STRING,
    defect_category STRING, is_defect BOOLEAN
    """,
)

dim_defect_df = (
    unknown_defect_df.unionByName(defect_members_df)
    .withColumn("_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

write_dimension(dim_defect_df, TABLES["dim_defect"], "defect_key")

In [0]:
# ===================================================
# BLOCK 9 — BUILD TEST-PROGRAM DIMENSION
# ===================================================

"""
Create distinct device/program-revision members so unit-test facts can be
analyzed across controlled program versions.
"""

program_members_df = (
    spark.table(TABLES["silver_tests"])
    .select("device_id", "program_revision")
    .filter(
        F.col("device_id").isNotNull()
        & F.col("program_revision").isNotNull()
    )
    .distinct()
    .select(
        F.xxhash64("device_id", "program_revision").alias("test_program_key"),
        "device_id",
        "program_revision",
    )
)

unknown_program_df = spark.createDataFrame(
    [(0, "UNKNOWN", "UNKNOWN")],
    "test_program_key LONG, device_id STRING, program_revision STRING",
)

dim_test_program_df = (
    unknown_program_df.unionByName(program_members_df)
    .withColumn("_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID))
    .withColumn("_gold_processed_at_utc", F.current_timestamp())
)

write_dimension(
    dim_test_program_df,
    TABLES["dim_test_program"],
    "test_program_key",
)

In [0]:
# ===================================================
# BLOCK 10 — INITIALIZE DEVICE SCD TYPE 2
# ===================================================

"""
Initialize one historical device member per Silver business key when the
SCD table does not yet exist.

The historical attributes are device name, product group, package type,
target yield, and lifecycle status.
"""

device_source_df = (
    spark.table(TABLES["silver_devices"])
    .select(
        "device_id",
        "device_name",
        "product_group_id",
        "package_type",
        "target_yield",
        "lifecycle_status",
    )
)

if not spark.catalog.tableExists(TABLES["dim_device_scd2"]):
    initial_device_df = (
        device_source_df
        .withColumn("effective_from", F.lit(HISTORY_START_DATE).cast("date"))
        .withColumn("effective_to", F.lit(OPEN_END_DATE).cast("date"))
        .withColumn("is_current", F.lit(True))
        .withColumn("version_number", F.lit(1))
        .withColumn(
            "device_key",
            F.xxhash64(
                "device_id",
                F.col("effective_from").cast("string"),
            ),
        )
        .withColumn("_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID))
        .withColumn("_gold_processed_at_utc", F.current_timestamp())
        .select(
            "device_key",
            "device_id",
            "device_name",
            "product_group_id",
            "package_type",
            "target_yield",
            "lifecycle_status",
            "effective_from",
            "effective_to",
            "is_current",
            "version_number",
            "_gold_pipeline_run_id",
            "_gold_processed_at_utc",
        )
    )

    unknown_device_df = spark.createDataFrame(
        [(
            0, "UNKNOWN", "Unknown Device", "UNKNOWN", "UNKNOWN",
            None, "UNKNOWN", HISTORY_START_DATE, OPEN_END_DATE, True, 1
        )],
        """
        device_key LONG, device_id STRING, device_name STRING,
        product_group_id STRING, package_type STRING,
        target_yield DECIMAL(9,6), lifecycle_status STRING,
        effective_from DATE, effective_to DATE, is_current BOOLEAN,
        version_number INT
        """,
    ).withColumn(
        "_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID)
    ).withColumn(
        "_gold_processed_at_utc", F.current_timestamp()
    )

    (
        unknown_device_df.unionByName(initial_device_df).write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TABLES["dim_device_scd2"])
    )

    print("Device SCD Type 2 dimension initialized.")
else:
    print("Existing device SCD Type 2 history retained.")

In [0]:
# ===================================================
# BLOCK 11 — INITIALIZE DEVICE SCD TYPE 2
# ===================================================

"""
Apply a controlled future-dated classification change for DV001.

The change demonstrates the production SCD pattern: expire the previous
current member and insert a new current version. The operation is
idempotent because the new effective-date version is inserted only once.
"""

current_device_df = (
    spark.table(TABLES["dim_device_scd2"])
    .filter(
        (F.col("device_id") == SIMULATED_DEVICE_ID)
        & F.col("is_current")
    )
)

assert current_device_df.count() == 1

change_exists = (
    spark.table(TABLES["dim_device_scd2"])
    .filter(
        (F.col("device_id") == SIMULATED_DEVICE_ID)
        & (F.col("effective_from") == F.lit(SIMULATED_CHANGE_DATE))
    )
    .count()
    > 0
)

if not change_exists:
    previous_row = current_device_df.first()

    spark.sql(
        f"""
        UPDATE {TABLES['dim_device_scd2']}
        SET
            effective_to = DATE_SUB(DATE('{SIMULATED_CHANGE_DATE}'), 1),
            is_current = FALSE,
            _gold_pipeline_run_id = '{PIPELINE_RUN_ID}',
            _gold_processed_at_utc = CURRENT_TIMESTAMP()
        WHERE device_id = '{SIMULATED_DEVICE_ID}'
          AND is_current = TRUE
        """
    )

    changed_device_df = spark.createDataFrame(
        [(
            previous_row["device_id"],
            previous_row["device_name"],
            previous_row["product_group_id"],
            previous_row["package_type"],
            previous_row["target_yield"],
            "MATURE",
            SIMULATED_CHANGE_DATE,
            OPEN_END_DATE,
            True,
            previous_row["version_number"] + 1,
        )],
        """
        device_id STRING, device_name STRING, product_group_id STRING,
        package_type STRING, target_yield DECIMAL(9,6),
        lifecycle_status STRING, effective_from DATE, effective_to DATE,
        is_current BOOLEAN, version_number INT
        """,
    ).withColumn(
        "device_key",
        F.xxhash64("device_id", F.col("effective_from").cast("string")),
    ).withColumn(
        "_gold_pipeline_run_id", F.lit(PIPELINE_RUN_ID)
    ).withColumn(
        "_gold_processed_at_utc", F.current_timestamp()
    ).select(
        "device_key",
        "device_id",
        "device_name",
        "product_group_id",
        "package_type",
        "target_yield",
        "lifecycle_status",
        "effective_from",
        "effective_to",
        "is_current",
        "version_number",
        "_gold_pipeline_run_id",
        "_gold_processed_at_utc",
    )

    (
        changed_device_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLES["dim_device_scd2"])
    )

    print("Controlled DV001 SCD Type 2 change applied.")
else:
    print("Controlled DV001 SCD version already exists; no rows changed.")

In [0]:
# ===================================================
# BLOCK 12 — FINAL DIMENSION SUMMARY
# ===================================================

"""
Publish dimension populations and confirm the controlled device-history
record before the separate validation notebook runs.
"""

for table_key in [
    "dim_date",
    "dim_site",
    "dim_product_group",
    "dim_equipment",
    "dim_defect",
    "dim_test_program",
    "dim_device_scd2",
]:
    print(f"{TABLES[table_key]}: {spark.table(TABLES[table_key]).count():,}")

display(
    spark.table(TABLES["dim_device_scd2"])
    .filter(F.col("device_id") == SIMULATED_DEVICE_ID)
    .orderBy("effective_from")
)

PIPELINE_END_TIME = datetime.now(timezone.utc)

print("GOLD DIMENSION PIPELINE COMPLETED")
print(f"Pipeline end UTC: {PIPELINE_END_TIME.isoformat()}")